### Importing the dataset

In [41]:
import json

DATA_PATH = "../data/raw/qa_data.json"

data = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print("Dataset loaded successfully!")
print("Number of examples:", len(data))

Dataset loaded successfully!
Number of examples: 10000


In [42]:
# Contents of the dataset (1 sample)
print(data[0])

{'knowledge': "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.", 'question': "Which magazine was started first Arthur's Magazine or First for Women?", 'right_answer': "Arthur's Magazine", 'hallucinated_answer': 'First for Women was started first.'}


In [43]:
example = data[0]

print("KNOWLEDGE:")
print(example["knowledge"])

print("\nQUESTION:")
print(example["question"])

print("\nRIGHT ANSWER:")
print(example["right_answer"])

print("\nHALLUCINATED ANSWER:")
print(example["hallucinated_answer"])

KNOWLEDGE:
Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.

QUESTION:
Which magazine was started first Arthur's Magazine or First for Women?

RIGHT ANSWER:
Arthur's Magazine

HALLUCINATED ANSWER:
First for Women was started first.


### Splitting the json lines data into Train/Validation/Test sets

In [44]:
from sklearn.model_selection import train_test_split

# Convert to a list of indices
indices = list(range(len(data)))

# 80% training, 20% temporary
train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

# Split the remaining 20% equally:
# 10% validation, 10% test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42
)

print("Training records:", len(train_indices))
print("Validation records:", len(val_indices))
print("Test records:", len(test_indices))

Training records: 8000
Validation records: 1000
Test records: 1000


In [45]:
# verifying the Overlap
print("Train ∩ Validation:", len(set(train_indices) & set(val_indices)))
print("Train ∩ Test:", len(set(train_indices) & set(test_indices)))
print("Validation ∩ Test:", len(set(val_indices) & set(test_indices)))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [46]:
def create_examples(indices):
    examples = []

    for idx in indices:
        item = data[idx]

        # Correct answer
        examples.append({
            "knowledge": item["knowledge"],
            "question": item["question"],
            "answer": item["right_answer"],
            "label": 0
        })

        # Hallucinated answer
        examples.append({
            "knowledge": item["knowledge"],
            "question": item["question"],
            "answer": item["hallucinated_answer"],
            "label": 1
        })

    return examples

train_data = create_examples(train_indices)
val_data = create_examples(val_indices)
test_data = create_examples(test_indices)

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Test examples:", len(test_data))

Training examples: 16000
Validation examples: 2000
Test examples: 2000


In [47]:
# Inspecting one sample
print(train_data[0])

{'knowledge': 'Vincenzo Salvatore Carmelo Francesco Bellini (] ; 3 November 1801 – 23 September 1835) was an Italian opera composer, who was known for his long-flowing melodic lines for which he was named "the Swan of Catania".Igor Fyodorovich Stravinsky (sometimes spelled "Strawinski", "Strawinsky", or "Stravinskii"; Ukrainian: І́гор Фе́дорович Страві́нський , ] ;Russian: И́горь Фёдорович Страви́нский , "Igorʹ Fëdorovič Stravinskij"; ] ; 17 June [O.S. 5 June] 1882 6 April 1971) was a Russian Empire-born composer, pianist, and conductor.', 'question': 'Who has knowledge of more arts, Vincenzo Bellini or Igor Stravinsky', 'answer': 'Igor Fyodorovich Stravinsky', 'label': 0}


In [48]:
print(train_data[1])

{'knowledge': 'Vincenzo Salvatore Carmelo Francesco Bellini (] ; 3 November 1801 – 23 September 1835) was an Italian opera composer, who was known for his long-flowing melodic lines for which he was named "the Swan of Catania".Igor Fyodorovich Stravinsky (sometimes spelled "Strawinski", "Strawinsky", or "Stravinskii"; Ukrainian: І́гор Фе́дорович Страві́нський , ] ;Russian: И́горь Фёдорович Страви́нский , "Igorʹ Fëdorovič Stravinskij"; ] ; 17 June [O.S. 5 June] 1882 6 April 1971) was a Russian Empire-born composer, pianist, and conductor.', 'question': 'Who has knowledge of more arts, Vincenzo Bellini or Igor Stravinsky', 'answer': 'Vincenzo Bellini had more knowledge.', 'label': 1}


In [49]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['knowledge', 'question', 'answer', 'label'],
    num_rows: 16000
})
Dataset({
    features: ['knowledge', 'question', 'answer', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['knowledge', 'question', 'answer', 'label'],
    num_rows: 2000
})


In [50]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: None")

PyTorch: 2.14.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 2050


In [51]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

#### Tokenizing 

In [52]:
example = train_dataset[0]

knowledge = example["knowledge"]
question = example["question"]
answer = example["answer"]

print("Knowledge:", knowledge)
print("\nQuestion:", question)
print("\nAnswer:", answer)

Knowledge: Vincenzo Salvatore Carmelo Francesco Bellini (] ; 3 November 1801 – 23 September 1835) was an Italian opera composer, who was known for his long-flowing melodic lines for which he was named "the Swan of Catania".Igor Fyodorovich Stravinsky (sometimes spelled "Strawinski", "Strawinsky", or "Stravinskii"; Ukrainian: І́гор Фе́дорович Страві́нський , ] ;Russian: И́горь Фёдорович Страви́нский , "Igorʹ Fëdorovič Stravinskij"; ] ; 17 June [O.S. 5 June] 1882 6 April 1971) was a Russian Empire-born composer, pianist, and conductor.

Question: Who has knowledge of more arts, Vincenzo Bellini or Igor Stravinsky

Answer: Igor Fyodorovich Stravinsky


In [53]:
encoding = tokenizer(
    knowledge,
    question,
    answer,
    truncation=True,
    max_length=192,
    padding="max_length"
)

print(encoding.keys())

KeysView({'input_ids': [101, 24712, 17485, 19443, 2080, 11400, 4330, 5498, 1006, 1033, 1025, 1017, 2281, 12410, 1516, 2603, 2244, 10150, 1007, 2001, 2019, 3059, 3850, 4543, 1010, 2040, 2001, 2124, 2005, 2010, 2146, 1011, 8577, 17187, 3210, 2005, 2029, 2002, 2001, 2315, 1000, 1996, 10677, 1997, 4937, 13241, 1000, 1012, 14661, 1042, 7677, 7983, 12303, 2358, 2527, 27704, 1006, 2823, 11479, 1000, 13137, 19880, 1000, 1010, 1000, 13137, 20894, 1000, 1010, 2030, 1000, 2358, 2527, 6371, 5488, 2072, 1000, 1025, 5969, 1024, 1213, 29741, 14150, 16856, 1199, 15290, 29742, 14150, 16856, 16198, 1196, 22919, 16856, 10260, 25529, 29760, 18947, 29747, 23742, 23925, 15414, 1010, 1033, 1025, 2845, 1024, 1188, 29741, 14150, 16856, 23742, 1199, 15290, 29742, 14150, 16856, 16198, 1196, 22919, 16856, 10260, 25529, 10325, 18947, 29747, 23925, 15414, 1010, 1000, 100, 7349, 14604, 7903, 2358, 2527, 6371, 5488, 3501, 1000, 1025, 1033, 1025, 2459, 2238, 1031, 1051, 1012, 1055, 1012, 1019, 2238, 1033, 7131, 1020, 

In [54]:
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
print(tokens[:100])
print("Number of tokens:", len(encoding["input_ids"]))
print(encoding["attention_mask"][:50])

['[CLS]', 'vincenzo', 'salvatore', 'carmel', '##o', 'francesco', 'bell', '##ini', '(', ']', ';', '3', 'november', '1801', '–', '23', 'september', '1835', ')', 'was', 'an', 'italian', 'opera', 'composer', ',', 'who', 'was', 'known', 'for', 'his', 'long', '-', 'flowing', 'melodic', 'lines', 'for', 'which', 'he', 'was', 'named', '"', 'the', 'swan', 'of', 'cat', '##ania', '"', '.', 'igor', 'f', '##yo', '##dor', '##ovich', 'st', '##ra', '##vinsky', '(', 'sometimes', 'spelled', '"', 'straw', '##inski', '"', ',', '"', 'straw', '##insky', '"', ',', 'or', '"', 'st', '##ra', '##vin', '##ski', '##i', '"', ';', 'ukrainian', ':', 'і', '##г', '##о', '##р', 'ф', '##е', '##д', '##о', '##р', '##ович', 'с', '##т', '##р', '##а', '##в', '##і', '##н', '##с', '##ь', '##к']
Number of tokens: 192
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [55]:
def tokenize_function(batch):

    # Combine knowledge and question
    context = [
        knowledge + " " + question
        for knowledge, question in zip(
            batch["knowledge"],
            batch["question"]
        )
    ]

    # Answer becomes the second BERT sequence
    return tokenizer(
        context,
        batch["answer"],
        truncation=True,
        max_length=192,
        padding=False
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization complete.")

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization complete.


In [56]:
print(tokenized_train)

# Removing original strings
columns_to_remove = [
    "knowledge",
    "question",
    "answer"
]

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_val = tokenized_val.remove_columns(columns_to_remove)
tokenized_test = tokenized_test.remove_columns(columns_to_remove)

Dataset({
    features: ['knowledge', 'question', 'answer', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16000
})


In [57]:
print(tokenized_train[0])

{'label': 0, 'input_ids': [101, 24712, 17485, 19443, 2080, 11400, 4330, 5498, 1006, 1033, 1025, 1017, 2281, 12410, 1516, 2603, 2244, 10150, 1007, 2001, 2019, 3059, 3850, 4543, 1010, 2040, 2001, 2124, 2005, 2010, 2146, 1011, 8577, 17187, 3210, 2005, 2029, 2002, 2001, 2315, 1000, 1996, 10677, 1997, 4937, 13241, 1000, 1012, 14661, 1042, 7677, 7983, 12303, 2358, 2527, 27704, 1006, 2823, 11479, 1000, 13137, 19880, 1000, 1010, 1000, 13137, 20894, 1000, 1010, 2030, 1000, 2358, 2527, 6371, 5488, 2072, 1000, 1025, 5969, 1024, 1213, 29741, 14150, 16856, 1199, 15290, 29742, 14150, 16856, 16198, 1196, 22919, 16856, 10260, 25529, 29760, 18947, 29747, 23742, 23925, 15414, 1010, 1033, 1025, 2845, 1024, 1188, 29741, 14150, 16856, 23742, 1199, 15290, 29742, 14150, 16856, 16198, 1196, 22919, 16856, 10260, 25529, 10325, 18947, 29747, 23925, 15414, 1010, 1000, 100, 7349, 14604, 7903, 2358, 2527, 6371, 5488, 3501, 1000, 1025, 1033, 1025, 2459, 2238, 1031, 1051, 1012, 1055, 1012, 1019, 2238, 1033, 7131, 102

In [58]:
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

sample = tokenized_train[0]

print(type(sample["input_ids"]))
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["label"])

<class 'torch.Tensor'>
torch.Size([192])
torch.Size([192])
tensor(0)


### Preparing BERT

In [59]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

Device: cuda
GPU: NVIDIA GeForce RTX 2050
GPU memory: 4.00 GB


In [60]:
model = model.to(device)
print("Model moved to:", device)

Model moved to: cuda


In [61]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

BATCH_SIZE = 8

# Dynamically pad each batch
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

train_loader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0
)

val_loader = DataLoader(
    tokenized_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0
)

test_loader = DataLoader(
    tokenized_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 2000
Validation batches: 250
Test batches: 250


In [62]:
batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([8, 182])
token_type_ids torch.Size([8, 182])
attention_mask torch.Size([8, 182])
labels torch.Size([8])


In [63]:
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    token_type_ids=batch["token_type_ids"]
)

print("Logits shape:", outputs.logits.shape)

Logits shape: torch.Size([8, 2])


### Training BERT

In [64]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# Learning rate
LEARNING_RATE = 2e-5

# Weight decay
WEIGHT_DECAY = 0.01

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

criterion = nn.CrossEntropyLoss()

print("Loss function:", criterion)

Optimizer: AdamW
Learning rate: 2e-05
Weight decay: 0.01
Loss function: CrossEntropyLoss()


In [65]:
# Evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
def calculate_metrics(predictions, labels):
    """
    Calculate classification metrics.

    Label 0 = correct answer
    Label 1 = hallucinated answer
    """

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


print("Metrics function ready.")

Metrics function ready.


In [66]:
def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler
):
    """
    Train BERT for one complete epoch
    using mixed precision.
    """

    model.train()

    total_loss = 0.0

    all_predictions = []
    all_labels = []

    for batch_idx, batch in enumerate(dataloader):

        # -------------------------
        # Move data to GPU
        # -------------------------

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        token_type_ids = batch["token_type_ids"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        # -------------------------
        # Clear gradients
        # -------------------------

        optimizer.zero_grad(set_to_none=True)

        # -------------------------
        # Mixed precision forward
        # -------------------------

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

        # -------------------------
        # Backpropagation
        # -------------------------

        scaler.scale(loss).backward()

        # -------------------------
        # Optimizer update
        # -------------------------

        scaler.step(optimizer)

        scaler.update()

        # -------------------------
        # Track loss
        # -------------------------

        total_loss += loss.item()

        # -------------------------
        # Predictions
        # -------------------------

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        # -------------------------
        # Progress
        # -------------------------

        if (batch_idx + 1) % 50 == 0:

            print(
                f"  Batch "
                f"{batch_idx + 1}/"
                f"{len(dataloader)} "
                f"| Loss: "
                f"{loss.item():.4f}"
            )

    # -------------------------
    # Metrics
    # -------------------------

    metrics = calculate_metrics(
        all_predictions,
        all_labels
    )

    metrics["loss"] = (
        total_loss / len(dataloader)
    )

    return metrics

Scaler for BERT

In [67]:
scaler = torch.amp.GradScaler("cuda")
print("Mixed precision scaler ready.")

Mixed precision scaler ready.


In [68]:
def evaluate(
    model,
    dataloader,
    criterion,
    device
):
    """
    Evaluate BERT using mixed precision.
    """

    model.eval()

    total_loss = 0.0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            token_type_ids = batch["token_type_ids"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            # Mixed precision inference
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )

                logits = outputs.logits

                loss = criterion(
                    logits,
                    labels
                )

            total_loss += loss.item()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    metrics = calculate_metrics(
        all_predictions,
        all_labels
    )

    metrics["loss"] = (
        total_loss / len(dataloader)
    )

    return metrics

### Smoke Test (Development Check Only)

This test uses 128 examples to verify that the training pipeline,
GPU, loss calculation, backpropagation, and optimizer work correctly.

The resulting model is NOT used for the final experiment.

In [69]:
smoke_scaler = torch.amp.GradScaler("cuda")
print("Smoke-test mixed precision scaler ready.")

Smoke-test mixed precision scaler ready.


In [70]:
from torch.utils.data import DataLoader, Subset

SMOKE_TEST_SIZE = 128

small_train_dataset = Subset(
    tokenized_train,
    range(SMOKE_TEST_SIZE)
)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0
)

print("Smoke-test examples:", len(small_train_dataset))
print("Smoke-test batches:", len(small_train_loader))
print("Starting smoke test...")

smoke_metrics = train_one_epoch(
    model=model,
    dataloader=small_train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scaler=smoke_scaler
)

print("\nSmoke test complete!")

print(f"Loss:      {smoke_metrics['loss']:.4f}")
print(f"Accuracy:  {smoke_metrics['accuracy']:.4f}")
print(f"Precision: {smoke_metrics['precision']:.4f}")
print(f"Recall:    {smoke_metrics['recall']:.4f}")
print(f"F1:        {smoke_metrics['f1']:.4f}")

Smoke-test examples: 128
Smoke-test batches: 16
Starting smoke test...

Smoke test complete!
Loss:      0.5714
Accuracy:  0.7500
Precision: 0.6951
Recall:    0.8906
F1:        0.7808


#### Reloading BERT (flushing smoke test weights)

In [71]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = model.to(device)

print("Fresh BERT model loaded.")
print("Device:", next(model.parameters()).device)
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print("Fresh optimizer created.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh BERT model loaded.
Device: cuda:0
Fresh optimizer created.


In [72]:
EPOCHS = 3

print("Training configuration")
print("----------------------")
print("Model:          bert-base-uncased")
print("Epochs:         ", EPOCHS)
print("Batch size:     ", BATCH_SIZE)
print("Learning rate:  ", LEARNING_RATE)
print("Max length:     256")
print("Device:         ", device)

Training configuration
----------------------
Model:          bert-base-uncased
Epochs:          3
Batch size:      8
Learning rate:   2e-05
Max length:     256
Device:          cuda


In [73]:
print("Training examples:", len(tokenized_train))
print("Training batches:", len(train_loader))
print("Batch size:", BATCH_SIZE)

Training examples: 16000
Training batches: 2000
Batch size: 8


In [74]:
batch = next(iter(train_loader))

print("Batch keys:")
print(batch.keys())

print("\nBatch shapes:")
for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

Batch keys:
KeysView({'input_ids': tensor([[  101,  1999,  7068,  ...,     0,     0,     0],
        [  101, 15820,  6097,  ...,     0,     0,     0],
        [  101,  1000,  3722,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  2249,  ...,     0,     0,     0],
        [  101,  2033,  6810,  ...,     0,     0,     0],
        [  101,  2771, 16329,  ...,  1998,  2814,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([1, 1, 1, 0, 1, 1, 1, 0])})

Batch shapes:
input_ids: (8, 171)
token_type_ids: (8, 171)
attention_mask: (8, 171

Speed Test (800 samples)

In [75]:
from torch.utils.data import Subset, DataLoader

SPEED_TEST_SIZE = 800

speed_test_dataset = Subset(
    tokenized_train,
    range(min(SPEED_TEST_SIZE, len(tokenized_train)))
)

speed_test_loader = DataLoader(
    speed_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0
)

print("Speed-test examples:", len(speed_test_dataset))
print("Speed-test batches:", len(speed_test_loader))

Speed-test examples: 800
Speed-test batches: 100


In [77]:
import time

print("Starting speed test...")

start_time = time.time()

speed_metrics = train_one_epoch(
    model=model,
    dataloader=speed_test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scaler=scaler
)

elapsed_time = time.time() - start_time

print("\nSpeed test complete!")
print("--------------------")
print(f"Time: {elapsed_time:.2f} seconds")
print(f"Time: {elapsed_time / 60:.2f} minutes")

print("\nMetrics:")
print(f"Loss:      {speed_metrics['loss']:.4f}")
print(f"Accuracy:  {speed_metrics['accuracy']:.4f}")
print(f"Precision: {speed_metrics['precision']:.4f}")
print(f"Recall:    {speed_metrics['recall']:.4f}")
print(f"F1:        {speed_metrics['f1']:.4f}")

Starting speed test...
  Batch 50/100 | Loss: 0.4688
  Batch 100/100 | Loss: 0.4011

Speed test complete!
--------------------
Time: 220.35 seconds
Time: 3.67 minutes

Metrics:
Loss:      0.5755
Accuracy:  0.6937
Precision: 0.6224
Recall:    0.9850
F1:        0.7628


In [ ]:
import numpy as np

lengths = [
    len(example["input_ids"])
    for example in tokenized_train
]

print("Token length statistics")
print("-----------------------")
print(f"Minimum:  {min(lengths)}")
print(f"Maximum:  {max(lengths)}")
print(f"Mean:     {np.mean(lengths):.2f}")
print(f"Median:   {np.median(lengths):.2f}")
print(f"90th %:   {np.percentile(lengths, 90):.0f}")
print(f"95th %:   {np.percentile(lengths, 95):.0f}")
print(f"99th %:   {np.percentile(lengths, 99):.0f}")

print("\nLengths above thresholds:")
print(f"> 128: {sum(x > 128 for x in lengths)}")
print(f"> 192: {sum(x > 192 for x in lengths)}")
print(f"> 256: {sum(x > 256 for x in lengths)}")

Token length statistics
-----------------------
Minimum:  33
Maximum:  256
Mean:     112.46
Median:   106.00
90th %:   162
95th %:   182
99th %:   228

Lengths above thresholds:
> 128: 4575
> 192: 538
> 256: 0


### Main BERT Test

In [ ]:
import time

training_history = []

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    start_time = time.time()

    # -------------------------
    # Training
    # -------------------------

    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        scaler=scaler
)

    # -------------------------
    # Validation
    # -------------------------

    val_metrics = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device
    )

    epoch_time = time.time() - start_time

    # -------------------------
    # Store results
    # -------------------------

    epoch_results = {
        "epoch": epoch + 1,

        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],

        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],

        "time_seconds": epoch_time
    }

    training_history.append(epoch_results)

    # -------------------------
    # Print results
    # -------------------------

    print("\nTraining:")
    print(f"  Loss:      {train_metrics['loss']:.4f}")
    print(f"  Accuracy:  {train_metrics['accuracy']:.4f}")
    print(f"  Precision: {train_metrics['precision']:.4f}")
    print(f"  Recall:    {train_metrics['recall']:.4f}")
    print(f"  F1:        {train_metrics['f1']:.4f}")

    print("\nValidation:")
    print(f"  Loss:      {val_metrics['loss']:.4f}")
    print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"  Precision: {val_metrics['precision']:.4f}")
    print(f"  Recall:    {val_metrics['recall']:.4f}")
    print(f"  F1:        {val_metrics['f1']:.4f}")

    print(f"\nEpoch time: {epoch_time / 60:.2f} minutes")


Epoch 1/3


In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

history_path = RESULTS_DIR / "bert_training_history.json"

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(
        training_history,
        f,
        indent=4
    )

print(f"Training history saved to: {history_path}")